# Audio transcription — free HF vs paid OpenAI

Folder: **`10_AUDIO_TRANSCRIPTION/`**.

| Section | How | Money |
|---------|-----|-------|
| 1 | Hugging Face **Whisper** on your machine | **Free** model weights |
| 2 | OpenAI **`gpt-4o-mini-transcribe`** API | **Paid** — OpenAI billing |

**Pattern:** code first → explanation under it.

### Colab vs laptop (Section 1)

| | Colab | Your laptop |
|--|-------|-------------|
| Free Whisper weights? | Yes | Yes |
| Must use Colab? | No | You can run here too |
| Why people use Colab | Free **GPU** → faster | CPU works, can be slower |

### Before you run

1. Set `AUDIO_PATH` to your mp3 (Colab Drive or local path)  
2. Section 1: `HF_TOKEN` optional · Section 2: `OPENAI_API_KEY` required  
3. Long audio: Whisper uses `return_timestamps=True`  
4. Colab Section 1: GPU (T4) recommended for speed  


## Shared setup

In [ ]:
# Point to your audio file
from pathlib import Path
import os

# Colab + Drive (edit filename/folder if needed):
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

AUDIO_PATH = "/content/drive/MyDrive/Colab Notebooks/mrrting.mp3"

# Laptop example (comment Drive block above, use this instead):
# AUDIO_PATH = "/home/soudas/PERSONAL/AI_LEARNING/AI_LEARNING/10_AUDIO_TRANSCRIPTION/mrrting.mp3"

print("AUDIO_PATH =", AUDIO_PATH)
print("Audio exists:", Path(AUDIO_PATH).exists())


### Explanation — audio path

`AUDIO_PATH` is the full path to your mp3.

- **Colab:** mount Drive, then use something like  
  `/content/drive/MyDrive/Colab Notebooks/mrrting.mp3`
- **Laptop:** use your local path instead (no `drive.mount`)

Both sections below use the same `AUDIO_PATH`.


---

## Section 1 — Free Hugging Face Whisper

Model: `openai/whisper-medium.en`  
(OpenAI trained it; weights are on the **HF Hub**, **Apache-2.0**, not gated, **free to download**.)

In [ ]:
!pip -q install transformers accelerate torchaudio

### Explanation — install (HF)

| Package | Why |
|---------|-----|
| `transformers` | ASR `pipeline` + Whisper |
| `accelerate` | GPU helpers |
| `torchaudio` | Often needed for audio loading |

In [ ]:
import torch
from transformers import pipeline

try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("HF_TOKEN: logged in")
    else:
        print("HF_TOKEN: not set (OK — Whisper is public)")
except Exception as e:
    print("HF login skipped:", e)

device = 0 if torch.cuda.is_available() else -1
print("device:", "cuda:0" if device == 0 else "cpu")

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    device=device,
)

# Long audio (>30s) needs return_timestamps=True for Whisper pipeline
hf_result = asr(
    AUDIO_PATH,
    return_timestamps=True,
)

print("=== HF Whisper (free weights) ===")
if isinstance(hf_result, dict):
    print(hf_result.get("text", hf_result))
else:
    print(hf_result)


### What is `isinstance`? (right under the print code)

`isinstance` is normal **Python** — not Hugging Face.

```python
isinstance(hf_result, dict)
```

Means: **“Is `hf_result` a dictionary?”**

| Result | Meaning |
|--------|---------|
| `True` | It’s a dict → use `.get("text")` for the transcript |
| `False` | Not a dict → just `print(hf_result)` |

```python
isinstance({"text": "hi"}, dict)  # True
isinstance("hi", dict)            # False
isinstance("hi", str)             # True
```

In our print block:

```python
if isinstance(hf_result, dict):
    print(hf_result.get("text", hf_result))  # print transcript string
else:
    print(hf_result)                         # print whatever else came back
```

With `return_timestamps=True`, Whisper often returns a **dict** with `"text"` + `"chunks"`.  
We only want to print `"text"`.


### Explanation — free HF path

```text
audio file  →  Whisper on YOUR Colab machine  →  text
```

| Piece | Meaning |
|-------|---------|
| `pipeline("automatic-speech-recognition")` | Task string = speech-to-text |
| `model="openai/whisper-medium.en"` | English Whisper medium from HF Hub |
| `device=0` | GPU if available; `-1` = CPU (slower) |
| `asr(AUDIO_PATH, return_timestamps=True)` | Transcribe (needed if audio is longer than ~30s) |

#### Is this free?

| | |
|--|--|
| Download / run Whisper weights | **Yes — free** (Apache-2.0, not gated) |
| OpenAI API bill | **No** — you are **not** calling `api.openai.com` |
| Colab | Free tier GPU/time limits still apply |

#### Why `return_timestamps=True`?

Whisper’s HF pipeline, for longer clips (officially **> 30 seconds**), wants timestamps enabled so it can process the audio in chunks.

```python
asr(AUDIO_PATH, return_timestamps=True)
```

Without it you get an error on longer `mrrting.mp3` files. Short clips may work without it; longer meetings need it.

First run downloads ~model weights (be patient).

```text
HF Whisper = free model + your compute
```

#### Printing the result (`isinstance` / `.get("text")`)

```python
print("=== HF Whisper (free weights) ===")
if isinstance(hf_result, dict):
    print(hf_result.get("text", hf_result))
else:
    print(hf_result)
```

That block only **prints the transcript** safely, because the return type can vary.

#### What is `isinstance`?

`isinstance(x, some_type)` is a normal **Python** check:

> “Is `x` this kind of value?”

```python
isinstance(hf_result, dict)   # True if hf_result is a dictionary
isinstance("hello", str)      # True
isinstance(42, dict)          # False
```

| Piece | Meaning |
|-------|---------|
| `isinstance` | Built-in Python function |
| `hf_result` | The thing we got from Whisper |
| `dict` | Dictionary type (`{"key": "value"}`) |

So:

```python
if isinstance(hf_result, dict):
    # treat it like a dict → use .get("text")
else:
    # not a dict → just print it
```

Not Hugging Face magic — plain Python type checking.


| Line | Meaning |
|------|---------|
| `print("=== HF Whisper...")` | Label so you know which section’s output this is |
| `isinstance(hf_result, dict)` | Is the result a dictionary? |
| `hf_result.get("text", hf_result)` | If yes, print the `"text"` field (the transcript) |
| `else: print(hf_result)` | If not a dict, print whatever came back |

With `return_timestamps=True`, Whisper often returns a **dict** like:

```python
{"text": "full transcript...", "chunks": [...]}
```

So we print `.get("text")` — not the whole dict (that would dump timestamp chunks too).

`.get("text", hf_result)` = use `"text"` if present; otherwise print the whole object.

#### Do I need Colab for free HF?

**No.** Free means the **weights cost $0**, not “must run in Colab”.

- **Laptop:** install `transformers`, point `AUDIO_PATH` at a local mp3, run the same Whisper code (CPU OK, slower).  
- **Colab:** same code + free GPU → faster for `whisper-medium.en`.

Paid OpenAI section always needs internet + API key (laptop or Colab).


---

## Section 2 — Paid OpenAI transcription API

Model: `gpt-4o-mini-transcribe`  
Uses **`OPENAI_API_KEY`** → shows on **OpenAI usage / billing**.

In [ ]:
!pip -q install openai

### Explanation — install (OpenAI)

Only the OpenAI Python client. No Whisper download on your machine.

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception as e:
    print("Colab secret skipped (use env OPENAI_API_KEY):", e)

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY (Colab Secret or env)"

openai_client = OpenAI()

with open(AUDIO_PATH, "rb") as audio_file:
    openai_result = openai_client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio_file,
    )

print("=== OpenAI gpt-4o-mini-transcribe (PAID) ===")
print(openai_result.text)

### Explanation — paid OpenAI path

```text
audio file  →  internet  →  OpenAI servers  →  text
                 ↑
           OPENAI_API_KEY (billed)
```

| Piece | Meaning |
|-------|---------|
| `OpenAI()` | Default cloud client (`api.openai.com`) |
| `audio.transcriptions.create(...)` | Speech-to-text API |
| `model="gpt-4o-mini-transcribe"` | OpenAI’s transcription model |
| `file=audio_file` | Your uploaded bytes |
| `openai_result.text` | Transcript string |

#### Is this free?

**No.** This is **paid API usage** on your OpenAI account.

| | |
|--|--|
| OpenAI usage page | **Yes — this call appears** |
| HF Whisper section above | Does **not** appear on OpenAI billing |

Usually cheaper than big chat models, but still not free.

```text
OpenAI transcribe = paid API + OpenAI compute
```

---

## Recap — free vs paid

| | Section 1 HF Whisper | Section 2 OpenAI API |
|--|----------------------|----------------------|
| Model id | `openai/whisper-medium.en` | `gpt-4o-mini-transcribe` |
| Where it runs | Your Colab GPU/CPU | OpenAI cloud |
| Key | `HF_TOKEN` optional | `OPENAI_API_KEY` required |
| Money | Free weights | **Billed** |
| Shows on OpenAI usage? | No | **Yes** |

```text
Want $0 API bill  →  Section 1 (HF Whisper)
Want hosted API   →  Section 2 (OpenAI) — expect a charge
```

Same audio file → compare both transcripts side by side.